# TP3 — Scheduling LTE (4G) : MAXCI vs PF vs DRR

Analyse des résultats exportés par `tp-export`. Complétez les `# TODO` et les cellules **Réponse**.

In [ ]:
# Environnement : rend visibles les bibliothèques de l'image (pandas, matplotlib) quel que soit le noyau choisi
import sys, glob
sys.path += glob.glob('/home/opp_env/.venv/lib/python3.*/site-packages')
import sys; sys.path.append('/tp/common')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from tpanalyse import load_scalars
plt.rcParams['figure.figsize'] = (10, 4.5)

def jain(x):
    """Indice d'équité de Jain : 1 = parfaitement équitable, 1/n = un seul servi."""
    x = np.asarray(x, dtype=float)
    return x.sum()**2 / (len(x) * (x**2).sum())

## 1. Débit par UE pour chaque ordonnanceur (config `Sched`, runs 0..2)
Après `tp-run Sched 0..2` puis `tp-export results/Sched sched.csv` :

In [ ]:
sca, iv = load_scalars('sched.csv')
thr = sca[sca.name == 'cbrReceivedThroughput:mean'].copy()
thr['ue'] = thr.module.str.extract(r'ue\[(\d+)\]').astype(int)
thr = thr.merge(iv[['sched']], left_on='run', right_index=True)
tab = thr.pivot_table(index='ue', columns='sched', values='value') * 8 / 1e6   # Mbit/s
tab.round(2)

In [ ]:
tab.plot.bar(width=0.8); plt.ylabel('Débit reçu (Mbit/s)'); plt.xlabel('UE (0 = le plus proche de l\'eNB, 9 = le plus loin)')
plt.title('Débit par UE selon l\'ordonnanceur'); plt.grid(axis='y', alpha=.3); plt.show()

**Q3.1** — Décrivez la forme de chaque histogramme. Qui est servi par MAXCI ? Par DRR ? Où se situe PF ?

## 2. Efficacité contre équité

In [ ]:
resume = pd.DataFrame({
    'Débit cellule (Mbit/s)': tab.sum(),
    'Débit min UE (Mbit/s)': tab.min(),
    'Indice de Jain': tab.apply(jain),
})
resume.round(3)

**Q3.2** — Classez les trois ordonnanceurs selon le débit total, puis selon l'équité. Pourquoi ne peut-on pas être premier sur les deux ? Que vaut Jain pour MAXCI et que signifie ce chiffre ?

**Q3.3** — Le CQI moyen par UE explique le comportement de MAXCI. Tracez `averageCqiDl:mean` en fonction de la distance (`distance:mean`).

In [ ]:
cqi = sca[sca.name == 'averageCqiDl:mean'].copy(); cqi['ue'] = cqi.module.str.extract(r'ue\[(\d+)\]').astype(int)
d = cqi.groupby('ue').value.mean().to_frame('cqi')
d['distance'] = 200 + 250 * d.index          # positions fixées dans omnetpp.ini : ue[k] à 200 + 250·k m de l'eNB
plt.plot(d.distance, d.cqi, 'o-'); plt.xlabel('Distance à l\'eNB (m)'); plt.ylabel('CQI moyen DL'); plt.grid(alpha=.3); plt.show()
print(d)

## 3. Montée en charge (config `Charge`, runs 0..11)
Après `tp-run Charge 0..11` puis `tp-export results/Charge charge.csv` :

In [ ]:
sca, iv = load_scalars('charge.csv')
thr = sca[sca.name == 'cbrReceivedThroughput:mean'].merge(iv[['sched','numUEs']], left_on='run', right_index=True)
g = thr.groupby(['sched','numUEs']).value
res = pd.DataFrame({'cellule_Mbps': g.sum()*8/1e6, 'jain': g.apply(jain)}).reset_index()
fig, ax = plt.subplots(1, 2)
for s, grp in res.groupby('sched'):
    ax[0].plot(grp.numUEs, grp.cellule_Mbps, 'o-', label=s); ax[1].plot(grp.numUEs, grp.jain, 'o-', label=s)
ax[0].set_ylabel('Débit cellule (Mbit/s)'); ax[1].set_ylabel('Indice de Jain')
for a in ax: a.set_xlabel('Nombre d\'UE'); a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.show()

**Q3.4** — Comment évoluent le débit total et l'équité avec le nombre d'UE pour chaque ordonnanceur ? Lequel « profite » de la diversité multi-utilisateur ? Expliquez.

## 4. Trafic mixte (optionnel, config `Mixte`)
**Q3.5** — Avec 5 UE VoIP et 5 UE CBR, comparez le délai VoIP (`voIPFrameDelay:mean`) sous les trois ordonnanceurs. Lequel protège le mieux la voix ? Que manque-t-il à ces trois ordonnanceurs pour garantir une QoS ?

In [ ]:
# TODO : charger mixte.csv, extraire voIPFrameDelay:mean par sched, tracer


## 5. Synthèse (10 lignes)
- Ordonnanceur à choisir pour un opérateur qui vend du débit maximal :
- Pour un opérateur qui garantit un service minimum à tous :
- Pourquoi PF est-il le choix par défaut des équipementiers :